In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

def load_final_model():
    """Load the final model with both adapters"""
    model_name = "Qwen/Qwen2.5-1.5B-Instruct"
    
    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        trust_remote_code=True
    )
    
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    
    # Load base model
    base_model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=True
    )
    
    # Load both adapters
    print("Loading both adapters...")
    model = PeftModel.from_pretrained(
        base_model,
        "./qwen1.5b-clinical-reasoning/final_adapters"
    )
    
    return model, tokenizer

def inference():
    model, tokenizer = load_final_model()
    
    print("="*80)
    print("MEDICAL DIAGNOSTIC ASSISTANT INFERENCE")
    print("="*80)
    print("The model will analyze clinical notes and provide:")
    print("1. Most suspected disease")
    print("2. Other diseases with significant risk")
    print("3. Diagnostic reasoning") 
    print("4. Precautions for the most suspected disease")
    print("\nType 'quit' to exit\n")
    
    while True:
        clinical_note = input("Enter clinical note: ").strip()
        
        if clinical_note.lower() in ['quit', 'exit', 'q']:
            break
        
        # Create comprehensive prompt
        instruction = """Analyze this clinical note and provide:
                        1. Most suspected disease
                        2. Other diseases with significant risk  
                        3. Diagnostic reasoning
                        4. Precautions for the most suspected disease

                        Please structure your response as follows:
                        **Most Suspected Disease:** [disease name]
                        **Other Significant Risks:** [list of other diseases]
                        **Diagnostic Reasoning:** [your reasoning chain]
                        **Precautions:** [precautions for the most suspected disease]"""
        
        prompt = f"<|im_start|>user\n{instruction}\n\nClinical Note:\n{clinical_note}<|im_end|>\n<|im_start|>assistant\n"
        
        # Tokenize
        inputs = tokenizer(prompt, return_tensors="pt", max_length=1024, truncation=True)
        
        # MOVE INPUTS TO GPU - THIS IS THE FIX
        inputs = {k: v.to(model.device) for k, v in inputs.items()}
        
        # Generate
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=512,
                temperature=0.7,
                do_sample=True,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
                repetition_penalty=1.1
            )
        
        # Decode response
        response = tokenizer.decode(outputs[0], skip_special_tokens=True)
        
        # Extract just the assistant's response
        if "assistant" in response:
            assistant_part = response.split("assistant")[-1].strip()
        else:
            assistant_part = response
        
        print(f"\nAssistant Response:")
        print("-" * 50)
        print(assistant_part)
        print("-" * 50)
        print()

if __name__ == "__main__":
    inference()

/home/joetheguide/Documents/dev/AI_ML_jupyter/env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
`torch_dtype` is deprecated! Use `dtype` instead!


Loading both adapters...
MEDICAL DIAGNOSTIC ASSISTANT INFERENCE
The model will analyze clinical notes and provide:
1. Most suspected disease
2. Other diseases with significant risk
3. Diagnostic reasoning
4. Precautions for the most suspected disease

Type 'quit' to exit



/home/joetheguide/Documents/dev/AI_ML_jupyter/env/lib/python3.13/site-packages/peft/peft_model.py:598: UserWarning: Found missing adapter keys while loading the checkpoint: ['base_model.model.model.layers.0.mlp.gate_proj.lora_A.default.weight', 'base_model.model.model.layers.0.mlp.gate_proj.lora_B.default.weight', 'base_model.model.model.layers.0.mlp.up_proj.lora_A.default.weight', 'base_model.model.model.layers.0.mlp.up_proj.lora_B.default.weight', 'base_model.model.model.layers.1.mlp.gate_proj.lora_A.default.weight', 'base_model.model.model.layers.1.mlp.gate_proj.lora_B.default.weight', 'base_model.model.model.layers.1.mlp.up_proj.lora_A.default.weight', 'base_model.model.model.layers.1.mlp.up_proj.lora_B.default.weight', 'base_model.model.model.layers.2.mlp.gate_proj.lora_A.default.weight', 'base_model.model.model.layers.2.mlp.gate_proj.lora_B.default.weight', 'base_model.model.model.layers.2.mlp.up_proj.lora_A.default.weight', 'base_model.model.model.layers.2.mlp.up_proj.lora_B.def


Assistant Response:
--------------------------------------------------
### Most Suspected Disease: Transient Ischemic Attack (TIA) or Stroke

**Other Significant Risks:**
- **Cerebral Thrombosis**: Due to the presence of hypertension.
- **Vascular Dementia**: Given the patient's age.
- **Cerebrovascular Accident (CVA)**: If symptoms persist longer than expected.

**Diagnostic Reasoning:**
The patient’s presentation includes sudden onset of right-sided weakness, slurred speech, right facial droop, and arm drift. These are classic signs of a stroke or TIA. The high blood pressure (BP 190/100 mmHg) significantly increases the risk of vascular events such as a stroke. Additionally, the absence of a headache and no history of prior trauma further supports the diagnosis of a cerebrovascular event.

**Precautions:**
- **Immediate Medical Attention**: Urgently seek medical evaluation to rule out a full-blown stroke.
- **Blood Pressure Control**: Start antihypertensive therapy immediately to r